In [7]:
# The purpose of this notebook is to try removing the outliers in the training data

In [8]:
# import data

from pathlib import Path
import pandas as pd

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'

df = pd.read_excel(excel_path)

print()

In [9]:
# Show the 25 original formulas and their three measured targets from df.
formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "Dry Strength Mean",
    "Wet Strength Mean",
]
raw_data_summary = df[formula_cols + measured_target_cols].copy()
raw_data_summary[formula_cols] = raw_data_summary[formula_cols].fillna(0.0)
raw_data_summary[measured_target_cols] = raw_data_summary[measured_target_cols].apply(
    pd.to_numeric, errors="raise"
)

assert len(raw_data_summary) == 25, (
    f"Expected the 25 original samples; found {len(raw_data_summary)}."
)
display(raw_data_summary)

,TEM,Joncryl,NC513,PEG-DE,NGDE,Avg_Density_g_cm3,Dry Strength Mean,Wet Strength Mean
0,8,0.0,0.0,0.0,0.0,0.192000,53.674733,23.680000
1,8,0.7,0.0,0.0,0.0,0.195000,68.863717,24.258481
2,8,1.0,0.0,0.0,0.0,0.159000,50.405845,21.329450
3,8,1.5,0.0,0.0,0.0,0.159933,54.717605,20.183592
4,8,2.0,0.0,0.0,0.0,0.201000,85.787785,27.585357
5,8,3.0,0.0,0.0,0.0,0.202000,80.535443,27.921901
6,5,0.0,0.0,0.0,0.0,0.298930,157.659876,51.133703
7,5,0.0,0.7,0.0,0.0,0.360035,215.989325,82.812461
8,5,0.0,1.0,0.0,0.0,0.355557,218.543779,77.134354
9,5,0.0,2.0,0.0,0.0,0.358055,196.341194,76.406460


In [ ]:
# Compare the baseline GPR with a wet-strength-only lower TEM length-scale bound.

import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feature_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
target_cols = [
    "Avg_Density_g_cm3",
    "Dry Strength Mean",
    "Wet Strength Mean",
]
target_labels = {
    "Avg_Density_g_cm3": "Average density (g/cm^3)",
    "Dry Strength Mean": "Dry strength",
    "Wet Strength Mean": "Wet strength",
}
wet_strength_target = "Wet Strength Mean"

# Blank ingredient cells mean that ingredient was not used in the formula.
model_data = df[feature_cols + target_cols].copy()
model_data[feature_cols] = model_data[feature_cols].fillna(0.0)
model_data[target_cols] = model_data[target_cols].apply(pd.to_numeric, errors="raise")

assert len(model_data) == 25, f"Expected all 25 samples; found {len(model_data)}."
assert not model_data[target_cols].isna().any().any(), "A target contains missing values."

X = model_data[feature_cols]
loo = LeaveOneOut()

def make_estimator(tem_minimum):
    length_scale_bounds = np.tile(
        np.array([[1e-2, 1e3]], dtype=float),
        (len(feature_cols), 1),
    )
    length_scale_bounds[feature_cols.index("TEM"), 0] = tem_minimum
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=np.ones(len(feature_cols)),
            length_scale_bounds=length_scale_bounds,
            nu=2.5,
        )
        + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
    )
    return make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=42,
        ),
    )


gpr_models = {}
baseline_models = {}
cv_predictions = pd.DataFrame(index=model_data.index)
results = []
warning_rows = []
wet_strength_predictions = {}

for target in target_cols:
    y = model_data[target]
    configurations = [("Baseline", 1e-2)]
    if target == wet_strength_target:
        configurations.append(("Lower TEM minimum", 1e-3))

    for configuration, tem_minimum in configurations:
        estimator = make_estimator(tem_minimum)
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            oof_prediction = cross_val_predict(
                clone(estimator), X, y, cv=loo, n_jobs=1
            )
            final_model = clone(estimator).fit(X, y)

        warning_rows.append({
            "target": target,
            "configuration": configuration,
            "warning_count": len(caught_warnings),
            "warnings": " | ".join(
                sorted({str(item.message) for item in caught_warnings})
            ),
        })
        results.append({
            "target": target,
            "configuration": configuration,
            "LOOCV R2": r2_score(y, oof_prediction),
            "LOOCV MAE": mean_absolute_error(y, oof_prediction),
            "LOOCV RMSE": np.sqrt(mean_squared_error(y, oof_prediction)),
            "optimized kernel (all 25 samples)": str(
                final_model.named_steps["gaussianprocessregressor"].kernel_
            ),
        })

        if configuration == "Baseline":
            baseline_models[target] = final_model
        else:
            gpr_models[target] = final_model

        if target != wet_strength_target or configuration == "Baseline":
            gpr_models.setdefault(target, final_model)
            cv_predictions[f"{target} - LOOCV prediction"] = oof_prediction

        if target == wet_strength_target:
            wet_strength_predictions[configuration] = oof_prediction
            if configuration == "Lower TEM minimum":
                cv_predictions[f"{target} - LOOCV prediction"] = oof_prediction

results_df = pd.DataFrame(results).set_index(["target", "configuration"])
print("LOOCV performance:")
display(results_df)

print("Wet-strength per-formula error change (negative is better):")
wet_strength_errors = model_data[feature_cols].copy()
wet_strength_errors["Measured"] = model_data[wet_strength_target]
wet_strength_errors["Baseline prediction"] = wet_strength_predictions["Baseline"]
wet_strength_errors["Baseline error"] = (
    wet_strength_errors["Baseline prediction"] - wet_strength_errors["Measured"]
)
wet_strength_errors["Lower-bound prediction"] = wet_strength_predictions[
    "Lower TEM minimum"
]
wet_strength_errors["Lower-bound error"] = (
    wet_strength_errors["Lower-bound prediction"] - wet_strength_errors["Measured"]
)
wet_strength_errors["Absolute error change"] = (
    wet_strength_errors["Lower-bound error"].abs()
    - wet_strength_errors["Baseline error"].abs()
)
display(wet_strength_errors.sort_values("Absolute error change"))

print("Convergence warnings by target and configuration:")
display(pd.DataFrame(warning_rows).drop(columns="warnings"))
for warning_row in warning_rows:
    if warning_row["warnings"]:
        print(
            f"{warning_row['target']} / {warning_row['configuration']}: "
            f"{warning_row['warnings']}"
        )

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# Compare held-out LOOCV predictions with the measured values.
fig, axes = plt.subplots(1, len(target_cols), figsize=(15, 4), squeeze=False)
for col, target in enumerate(target_cols):
    axis = axes[0, col]
    actual = model_data[target].to_numpy()
    predicted = cv_predictions[f"{target} - LOOCV prediction"].to_numpy()
    lower = min(actual.min(), predicted.min())
    upper = max(actual.max(), predicted.max())
    padding = (upper - lower) * 0.05 or 1.0
    limits = (lower - padding, upper + padding)
    axis.scatter(actual, predicted, color="tab:blue", edgecolor="white", zorder=3)
    axis.plot(limits, limits, linestyle="--", color="black", linewidth=1)
    axis.set_xlim(limits)
    axis.set_ylim(limits)
    axis.set_title(f"{target_labels[target]}\nLOOCV R^2 = {r2_score(actual, predicted):.3f}")
    axis.set_xlabel("Measured value")
    axis.set_ylabel("LOOCV predicted value")
    axis.grid(alpha=0.2)
fig.suptitle("Measured vs. out-of-fold predictions")
fig.tight_layout()
plt.show()


# Select formulas where Joncryl is the only ingredient that varies.
fixed_cols = [col for col in feature_cols if col != "Joncryl"]
fixed_values = X.iloc[0][fixed_cols]
same_formula = np.isclose(X[fixed_cols], fixed_values).all(axis=1)
joncryl_data = X.loc[same_formula]

if len(joncryl_data) != 6:
    raise ValueError(
        f"Expected 6 formulas with only Joncryl varying; found {len(joncryl_data)}."
    )

baseline_formula = joncryl_data.iloc[0].copy()
print("Fixed ingredient values:")
display(fixed_values.to_frame("Fixed value"))

n_grid = 100
n_posterior_samples = 5
rng = np.random.default_rng(42)
joncryl_grid = np.linspace(0.0, 6.0, n_grid)

grid_data = pd.DataFrame(
    np.repeat(baseline_formula.to_numpy()[None, :], n_grid, axis=0),
    columns=feature_cols,
)
grid_data["Joncryl"] = joncryl_grid

fig, axes = plt.subplots(1, len(target_cols), figsize=(15, 4), squeeze=False)

for col, target in enumerate(target_cols):
    axis = axes[0, col]
    pipeline = gpr_models[target]
    scaled_grid = pipeline.named_steps["standardscaler"].transform(grid_data)
    gpr = pipeline.named_steps["gaussianprocessregressor"]
    mean, covariance = gpr.predict(scaled_grid, return_cov=True)

    target_scale = model_data[target].std(ddof=0)
    if target_scale == 0:
        target_scale = 1.0
    observation_noise = gpr.kernel_.k2.noise_level * target_scale**2
    covariance -= observation_noise * np.eye(n_grid)
    covariance = (covariance + covariance.T) / 2

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    covariance_factor = eigenvectors * np.sqrt(np.clip(eigenvalues, 0, None))
    posterior_samples = (
        mean[:, None]
        + covariance_factor @ rng.standard_normal((n_grid, n_posterior_samples))
    )

    for sample_index in range(n_posterior_samples):
        axis.plot(
            joncryl_grid,
            posterior_samples[:, sample_index],
            color="tab:green",
            alpha=0.20,
            linewidth=2,
        )

    posterior_std = np.sqrt(np.clip(np.diag(covariance), 0, None))
    axis.fill_between(
        joncryl_grid,
        mean - 1.96 * posterior_std,
        mean + 1.96 * posterior_std,
        color="tab:orange",
        alpha=0.18,
        label="95% posterior interval",
    )
    axis.plot(
        joncryl_grid, mean, color="tab:orange", linewidth=2, label="Posterior mean"
    )
    axis.scatter(
        joncryl_data["Joncryl"],
        model_data.loc[joncryl_data.index, target],
        color="black",
        edgecolor="white",
        linewidth=0.5,
        s=50,
        zorder=5,
        label="Measured formulas",
    )

    axis.set_title(target_labels[target])
    axis.set_xlabel("Joncryl amount")
    axis.set_ylabel(target_labels[target])
    axis.grid(alpha=0.2)
    if col == 0:
        axis.legend(loc="best", fontsize=8)

fig.suptitle("GPR posterior slices: varying Joncryl only")
fig.tight_layout()
plt.show()